# Portable section to make it compatible with local and colab

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("google.colab unavailable in this runtime; falling back to local paths.")
        IN_COLAB = False

PROJECT_ROOT = "/content/drive/MyDrive/TheRealDNNExp" if IN_COLAB else os.path.abspath("..")
MODULES_DIR = os.path.join(PROJECT_ROOT, "Problem2", "modules")
RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "Problem2", "figures")

if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)

print(f"Project root: {PROJECT_ROOT}")
print(f"Modules dir: {MODULES_DIR}")
print(f"Results CSV path: {RESULTS_CSV_PATH}")
print(f"Figures dir: {FIGURES_DIR}")

Project root: c:\NileshLocal\Repo\MyExpWithML\TheRealDNNExp
Modules dir: c:\NileshLocal\Repo\MyExpWithML\TheRealDNNExp\Problem2\modules
Results CSV path: c:\NileshLocal\Repo\MyExpWithML\TheRealDNNExp\Results\results.csv
Figures dir: c:\NileshLocal\Repo\MyExpWithML\TheRealDNNExp\Problem2\figures


# Problem 2: Transformer from Scratch
Student: Nilesh Ranjan Mitra | ID: 2026aj05496
d=128 | heads=4 | layers=4 | Seeds: 42, 123, 999

## Step 2 — Attention from Scratch (P2.1)
Scaled dot-product attention with optional mask, verified against
torch.nn.functional.scaled_dot_product_attention, plus a causal-mask
test and a hand-computed 3-token example (no instructor example given,
per Prof's second instruction — values chosen here).

In [2]:
# Unit tests: softmax rows sum to 1, matches PyTorch, causal mask
import torch
import torch.nn.functional as F
from attention import scaled_dot_product_attention, causal_mask

torch.manual_seed(0)
B, H, T, D_K = 2, 1, 5, 8
q = torch.randn(B, H, T, D_K)
k = torch.randn(B, H, T, D_K)
v = torch.randn(B, H, T, D_K)

# Test 1: each row of weights sums to 1
_, weights = scaled_dot_product_attention(q, k, v, return_weights=True)
row_sums = weights.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), "attention rows must sum to 1"

# Test 2: matches PyTorch's reference implementation within 1e-5
our_out = scaled_dot_product_attention(q, k, v)
ref_out = F.scaled_dot_product_attention(q, k, v)
max_err = (our_out - ref_out).abs().max().item()
assert max_err < 1e-5, f"mismatch vs PyTorch: {max_err}"
print(f"Max abs error vs F.scaled_dot_product_attention: {max_err:.2e}")

# Test 3: causal mask gives zero weight to future positions
mask = causal_mask(T)
_, causal_weights = scaled_dot_product_attention(q, k, v, mask=mask, return_weights=True)
upper_triangle = causal_weights.triu(diagonal=1)
assert torch.allclose(upper_triangle, torch.zeros_like(upper_triangle), atol=1e-6), "future positions must have zero weight"

# Causal version should also match PyTorch's own causal path
ref_causal_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
our_causal_out = scaled_dot_product_attention(q, k, v, mask=mask)
causal_err = (our_causal_out - ref_causal_out).abs().max().item()
assert causal_err < 1e-5, f"causal mismatch vs PyTorch: {causal_err}"
print(f"Max abs error (causal) vs PyTorch: {causal_err:.2e}")

print("All attention unit tests passed.")

Max abs error vs F.scaled_dot_product_attention: 2.38e-07
Max abs error (causal) vs PyTorch: 1.19e-07
All attention unit tests passed.


### Hand-computed 3-token example (d_k = 2)

Q = K =
[[1, 0],
 [0, 1],
 [1, 1]]

V =
[[1,   0  ],
 [0,   1  ],
 [0.5, 0.5]]

Scale = 1 / sqrt(2) = 0.7071

**Row 1** (q1=[1,0]): scores = [1, 0, 1] * 0.7071 = [0.7071, 0, 0.7071]
softmax -> [0.4011, 0.1978, 0.4011]
output = 0.4011*[1,0] + 0.1978*[0,1] + 0.4011*[0.5,0.5] = **[0.6017, 0.3983]**

**Row 2** (q2=[0,1]): scores = [0, 1, 1] * 0.7071 = [0, 0.7071, 0.7071]
softmax -> [0.1978, 0.4011, 0.4011]
output = **[0.3983, 0.6017]** (by symmetry with row 1)

**Row 3** (q3=[1,1]): scores = [1, 1, 2] * 0.7071 = [0.7071, 0.7071, 1.4142]
softmax -> [0.2482, 0.2482, 0.5035]
output = 0.2482*[1,0] + 0.2482*[0,1] + 0.5035*[0.5,0.5] ≈ **[0.5000, 0.5000]**

In [3]:
# Verifying the hand calculation in code
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = Q.clone()
V = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

hand_output = torch.tensor([
    [0.6017, 0.3983],
    [0.3983, 0.6017],
    [0.5000, 0.5000],
])

code_output = scaled_dot_product_attention(Q, K, V)
max_err = (code_output - hand_output).abs().max().item()
assert max_err < 1e-3, f"hand vs code mismatch: {max_err}"
print("Code output:\n", code_output)
print(f"Max abs error vs hand calculation: {max_err:.2e}")
print("Hand calculation and code agree.")

Code output:
 tensor([[0.6017, 0.3983],
        [0.3983, 0.6017],
        [0.5000, 0.5000]])
Max abs error vs hand calculation: 3.19e-05
Hand calculation and code agree.


## Step 3 — Multi-Head Attention and Pre-LN Encoder Block (P2.2)
W_Q/W_K/W_V/W_O with head reshaping (built on the Task 1 attention core),
a pre-LayerNorm encoder block with a 4d-wide feed-forward layer, residual
connections and dropout, and sinusoidal/learned/none positional encodings.
Parameter count is derived by hand as 4d^2 + 2*d*d_ff per block and checked
against sum(p.numel()).

In [4]:
# import + shape test for MultiHeadAttention:

from multihead_attention import MultiHeadAttention

torch.manual_seed(0)
D_MODEL, N_HEADS, SEQ_LEN, BATCH = 128, 4, 7, 3
mha = MultiHeadAttention(d_model=D_MODEL, n_heads=N_HEADS)
x = torch.randn(BATCH, SEQ_LEN, D_MODEL)

# Shape must be preserved: (B, T, d_model) -> (B, T, d_model)
out = mha(x)
assert out.shape == (BATCH, SEQ_LEN, D_MODEL), f"unexpected shape {out.shape}"

# Causal masking inside MHA: with a causal mask, changing a future token
# must not change an earlier position's output.
from attention import causal_mask
mask = causal_mask(SEQ_LEN)
out_causal_1 = mha(x, mask=mask)
x_perturbed = x.clone()
x_perturbed[:, -1, :] += 10.0  # perturb only the LAST position
out_causal_2 = mha(x_perturbed, mask=mask)
assert torch.allclose(out_causal_1[:, :-1, :], out_causal_2[:, :-1, :], atol=1e-5), \
    "causal MHA: earlier positions must be unaffected by a later-position perturbation"

print("MultiHeadAttention shape + causal-independence tests passed.")

MultiHeadAttention shape + causal-independence tests passed.


In [5]:
# import + shape test for positional encodings:

from positional_encoding import (
    SinusoidalPositionalEncoding, LearnedPositionalEncoding, NoPositionalEncoding
)

pe_sin = SinusoidalPositionalEncoding(d_model=D_MODEL)
pe_learned = LearnedPositionalEncoding(d_model=D_MODEL)
pe_none = NoPositionalEncoding()

x = torch.zeros(BATCH, SEQ_LEN, D_MODEL)

# "none" must be a true identity
assert torch.equal(pe_none(x), x), "NoPositionalEncoding must be identity"

# sinusoidal PE values must lie in [-1, 1] (sums of sin/cos)
out_sin = pe_sin(x)
assert out_sin.abs().max() <= 1.0 + 1e-6, "sinusoidal PE values should be bounded by 1"

# sinusoidal PE must differ across positions (so the model can tell them apart)
assert not torch.allclose(out_sin[:, 0, :], out_sin[:, 1, :]), \
    "adjacent positions must have distinct encodings"

# learned PE has trainable parameters; sinusoidal/none do not
assert sum(p.numel() for p in pe_learned.parameters()) == 5000 * D_MODEL
assert sum(p.numel() for p in pe_sin.parameters()) == 0
assert sum(p.numel() for p in pe_none.parameters()) == 0

print("Positional encoding tests passed (none=identity, sinusoidal bounded+distinct, learned has params).")

Positional encoding tests passed (none=identity, sinusoidal bounded+distinct, learned has params).


In [6]:
#  import + shape/residual test for EncoderBlock:

from encoder import EncoderBlock, TransformerEncoder

D_FF = 512
block = EncoderBlock(d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=0.0)
block.eval()  # dropout off for a deterministic shape/residual check

x = torch.randn(BATCH, SEQ_LEN, D_MODEL)
out = block(x)
assert out.shape == x.shape, f"encoder block must preserve shape, got {out.shape}"

# Residual sanity: zeroing out the MHA+FFN weights should make the block an identity
# (since Pre-LN output = x + sublayer(LN(x)), and sublayer(anything)=0 if W_O/last-linear=0)
with torch.no_grad():
    block.mha.W_O.weight.zero_(); block.mha.W_O.bias.zero_()
    block.ff[-1].weight.zero_(); block.ff[-1].bias.zero_()
identity_out = block(x)
assert torch.allclose(identity_out, x, atol=1e-5), \
    "with zeroed output projections, pre-LN block must reduce to identity (residual-only)"

print("EncoderBlock shape + residual-identity tests passed.")

EncoderBlock shape + residual-identity tests passed.


In [7]:
# parameter-count derivation vs sum(p.numel()):
# Hand derivation for d=128, d_ff=512 (one EncoderBlock):
d, d_ff = 128, 512

weights_only = 4 * d**2 + 2 * d * d_ff          # W_Q,W_K,W_V,W_O + FFN linear1+linear2 weights
mha_biases = 4 * d                              # b_Q, b_K, b_V, b_O
ffn_biases = d_ff + d                           # linear1 bias (d_ff,) + linear2 bias (d,)
ln_params = 2 * (2 * d)                         # 2 LayerNorms x (weight + bias), each (d,)

derived_total = weights_only + mha_biases + ffn_biases + ln_params

fresh_block = EncoderBlock(d_model=d, n_heads=4, d_ff=d_ff)
actual_total = sum(p.numel() for p in fresh_block.parameters())

print(f"Weights-only (assignment formula 4d^2 + 2*d*d_ff): {weights_only:,}")
print(f"  + MHA biases (4d):                               {mha_biases:,}")
print(f"  + FFN biases (d_ff + d):                          {ffn_biases:,}")
print(f"  + LayerNorm gamma/beta (2 x 2d):                  {ln_params:,}")
print(f"Derived grand total:                                {derived_total:,}")
print(f"Actual sum(p.numel()) on a real EncoderBlock:       {actual_total:,}")

assert weights_only == 196_608, f"expected 196,608 per the assignment, got {weights_only}"
assert derived_total == actual_total, \
    f"hand derivation ({derived_total}) must match sum(p.numel()) ({actual_total})"

print("Parameter-count reconciliation passed: 196,608 weights-only + 1,664 (biases+LN) = "
      f"{actual_total:,} total trainable params per block.")

Weights-only (assignment formula 4d^2 + 2*d*d_ff): 196,608
  + MHA biases (4d):                               512
  + FFN biases (d_ff + d):                          640
  + LayerNorm gamma/beta (2 x 2d):                  512
Derived grand total:                                198,272
Actual sum(p.numel()) on a real EncoderBlock:       198,272
Parameter-count reconciliation passed: 196,608 weights-only + 1,664 (biases+LN) = 198,272 total trainable params per block.


In [8]:
#  end-to-end TransformerEncoder stack sanity (4 layers, matches your d/heads/layers config):
# Matches the Configuration section of the README: d=128, heads=4, layers=4
encoder_stack = TransformerEncoder(d_model=128, n_heads=4, d_ff=512, n_layers=4, dropout=0.1)
total_stack_params = sum(p.numel() for p in encoder_stack.parameters())
print(f"4-layer encoder stack total trainable params: {total_stack_params:,}")

x = torch.randn(2, 10, 128)
out = encoder_stack(x)
assert out.shape == x.shape

# Causal variant works end-to-end too (needed for Task B later)
causal_out = encoder_stack(x, mask=causal_mask(10))
assert causal_out.shape == x.shape
print("TransformerEncoder stack forward pass (bidirectional and causal) passed.")

4-layer encoder stack total trainable params: 793,344
TransformerEncoder stack forward pass (bidirectional and causal) passed.
